## Setup

Imports and environment. `HF_HOME` redirects the Hugging Face model cache to scratch storage so 8B-parameter weights don't fill the home quota.

In [1]:
import os
os.environ["HF_HOME"] = "/rds/general/user/rmw324/home/raels_playground/hugging_face_llms"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import gc, re
import numpy as np
import pandas as pd
import torch

import dllm
from dllm.core.samplers import MDLMSampler, MDLMSamplerConfig

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.6.0+cu124 | cuda: True


## Load data

Read the PetFinder training table and the breed/colour lookup tables. `to_readable` swaps numeric codes (Gender=1, MaturitySize=2, …) for human-readable strings — needed because the language models expect English, not category IDs.

In [ ]:
DATA_DIR = "/rds/general/user/rmw324/home/raels_playground/datasets/pet_finder"

train = pd.read_csv(os.path.join(DATA_DIR, "train/train.csv"))
breed_labels = pd.read_csv(os.path.join(DATA_DIR, "BreedLabels.csv"))
color_labels = pd.read_csv(os.path.join(DATA_DIR, "ColorLabels.csv"))

breed_map = dict(zip(breed_labels["BreedID"], breed_labels["BreedName"]))
color_map = dict(zip(color_labels["ColorID"], color_labels["ColorName"]))

GENDER_MAP = {1: "Male", 2: "Female", 3: "Mixed"}
SIZE_MAP   = {1: "Small", 2: "Medium", 3: "Large", 4: "Extra Large"}
FUR_MAP    = {1: "Short", 2: "Medium", 3: "Long"}
YESNO_MAP  = {1: "Yes", 2: "No", 3: "Not Sure"}
HEALTH_MAP = {1: "Healthy", 2: "Minor Injury", 3: "Serious Injury"}
TYPE_MAP   = {1: "Dog", 2: "Cat"}

def to_readable(df):
    out = df.copy()
    out["Type"]         = out["Type"].map(TYPE_MAP)
    out["Gender"]       = out["Gender"].map(GENDER_MAP)
    out["MaturitySize"] = out["MaturitySize"].map(SIZE_MAP)
    out["FurLength"]    = out["FurLength"].map(FUR_MAP)
    out["Vaccinated"]   = out["Vaccinated"].map(YESNO_MAP)
    out["Dewormed"]     = out["Dewormed"].map(YESNO_MAP)
    out["Sterilized"]   = out["Sterilized"].map(YESNO_MAP)
    out["Health"]       = out["Health"].map(HEALTH_MAP)
    out["Breed1"]       = out["Breed1"].map(breed_map).fillna("Unknown")
    out["Breed2"]       = out["Breed2"].map(breed_map).fillna("None")
    out["Color1"]       = out["Color1"].map(color_map).fillna("Unknown")
    out["Color2"]       = out["Color2"].map(color_map).fillna("None")
    out["Color3"]       = out["Color3"].map(color_map).fillna("None")
    return out

print(train.shape)
train.head(2)

(14993, 24)


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,...,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,...,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0,0


## Pick descriptive rows

Keep 25 rows whose `Description` is long enough and contains words that hint at hidden fields ("he/she", colours, sizes, "vaccinated", …). These cues give the language models something to actually reason from when fields are masked.

In [76]:
CUE = re.compile(
    r"\b(?:he|she|him|her|his|hers|male|female|boy|girl|"
    r"black|white|brown|gray|grey|orange|yellow|tabby|cream|golden|"
    r"small|medium|large|tiny|huge|big|"
    r"short|long|fluffy|"
    r"vaccinated|sterilized|spayed|neutered|dewormed|kitten|puppy)\b",
    re.IGNORECASE,
)

mask = (
    train["Description"].notna()
    & (train["Description"].str.len() > 200)
    & train["Description"].str.contains(CUE, na=False, regex=True)
)
candidates = train[mask].head(25).reset_index(drop=True)
candidates_readable = to_readable(candidates)
print(f"Selected {len(candidates_readable)} rows")
candidates_readable[["Type","Name","Gender","Color1","MaturitySize","FurLength","Age"]].head()

Selected 25 rows


,Type,Name,Gender,Color1,MaturitySize,FurLength,Age
0,Cat,Nibble,Male,Black,Small,Short,3
1,Dog,Brisco,Male,Brown,Medium,Medium,1
2,Dog,Hunter,Male,Black,Medium,Short,1
3,Cat,BULAT,Male,Black,Medium,Long,12
4,Cat,Kitty,Female,Black,Medium,Medium,12


## Inspect candidates

Print every selected row in full so we can manually decide which cells to hide in the next step.

In [77]:
DISPLAY_FIELDS = ["Type","Name","Age","Breed1","Gender","Color1",
                  "MaturitySize","FurLength","Vaccinated","Sterilized",
                  "Health","Description"]

for i, row in candidates_readable.iterrows():
    print(f"\n=== Row {i} ===")
    for f in DISPLAY_FIELDS:
        val = str(row[f])
        if f == "Description" and len(val) > 500:
            val = val[:500] + "..."
        print(f"  {f}: {val}")


=== Row 0 ===
  Type: Cat
  Name: Nibble
  Age: 3
  Breed1: Tabby
  Gender: Male
  Color1: Black
  MaturitySize: Small
  FurLength: Short
  Vaccinated: No
  Sterilized: No
  Health: Healthy
  Description: Nibble is a 3+ month old ball of cuteness. He is energetic and playful. I rescued a couple of cats a few months ago but could not get them neutered in time as the clinic was fully scheduled. The result was this little kitty. I do not have enough space and funds to care for more cats in my household. Looking for responsible people to take over Nibble's care.

=== Row 1 ===
  Type: Dog
  Name: Brisco
  Age: 1
  Breed1: Mixed Breed
  Gender: Male
  Color1: Brown
  MaturitySize: Medium
  FurLength: Medium
  Vaccinated: Yes
  Sterilized: No
  Health: Healthy
  Description: Their pregnant mother was dumped by her irresponsible owner at the roadside near some shops in Subang Jaya. Gave birth to them at the roadside. They are all healthy and adorable puppies. Already dewormed, vaccinated and

## Choose what to mask

`manual_masks` says which fields to hide per row. These are the cells the models will be asked to recover. Only the listed columns (Gender, Color1, …) are eligible.

In [78]:
# Edit this dict after inspecting the rows above. Pick cells whose Description
# (or surrounding row context) plausibly cues the answer.
# Eligible columns: Gender, Color1, MaturitySize, FurLength, Vaccinated, Sterilized, Health, Age
manual_masks = {
    0:  ["Gender", "MaturitySize"],
    1:  ["Color1", "FurLength"],
    2:  ["Gender"],
    3:  ["MaturitySize", "Vaccinated"],
    4:  ["FurLength"],
    5:  ["Gender", "Age"],
    6:  ["Color1"],
    7:  ["MaturitySize"],
    8:  ["Gender", "FurLength"],
    9:  ["Color1"],
    10: ["Vaccinated"],
    11: ["MaturitySize"],
    12: ["Gender"],
    13: ["FurLength", "Color1"],
    14: ["Gender"],
    15: ["MaturitySize"],
    16: ["FurLength"],
    17: ["Color1"],
    18: ["Gender", "Vaccinated"],
    19: ["MaturitySize"],
    20: ["Gender"],
    21: ["Color1"],
    22: ["FurLength"],
    23: ["MaturitySize"],
    24: ["Gender"],
}

ELIGIBLE = {"Gender","Color1","MaturitySize","FurLength",
            "Vaccinated","Sterilized","Health","Age"}
for k, v in manual_masks.items():
    bad = set(v) - ELIGIBLE
    assert not bad, f"Row {k}: column(s) {bad} not eligible"
print(f"Total masked cells: {sum(len(v) for v in manual_masks.values())}")

Total masked cells: 32


## Render helpers

`render_for_llada` flattens the rows into one text blob, replacing each masked cell with `[MASK]` tokens sized to fit the original value. `find_mask_runs` later locates the contiguous mask spans in the tokenised input so predictions can be sliced back out per cell.

In [79]:
RENDER_FIELDS = ["Type","Name","Age","Breed1","Breed2","Gender",
                 "Color1","Color2","Color3","MaturitySize","FurLength",
                 "Vaccinated","Dewormed","Sterilized","Health","Description"]

def render_for_llada(rows_df, masks_by_row, tokenizer, max_desc_words=40):
    """Concatenate all rows into one text block. Masked cells are replaced
    by tokenizer.mask_token * n_tokens (sized to the original value).
    Returns (text, ordered_keys) — keys are (row_idx, col_name, original_value)
    in the same order their masks appear in the text."""
    mask_token = tokenizer.mask_token
    parts = []
    keys = []
    for i, row in rows_df.iterrows():
        parts.append(f"\n=== Row {i+1} ===\n")
        masked_cols = set(masks_by_row.get(i, []))
        for f in RENDER_FIELDS:
            val = str(row[f])
            if f == "Age":
                try: val = str(int(float(val)))
                except: pass
            if f == "Description":
                words = val.split()
                if len(words) > max_desc_words:
                    val = " ".join(words[:max_desc_words]) + "..."
            if f in masked_cols:
                n_tok = max(1, len(tokenizer.encode(val, add_special_tokens=False)))
                parts.append(f"{f}: {mask_token * n_tok}\n")
                keys.append((i, f, val))
            else:
                parts.append(f"{f}: {val}\n")
    return "".join(parts), keys

def find_mask_runs(input_ids, mask_id):
    """Return list of (start, end_exclusive) for each contiguous run of mask_id."""
    runs = []
    in_run = False
    s = 0
    for idx, t in enumerate(input_ids):
        if t == mask_id:
            if not in_run:
                s = idx; in_run = True
        else:
            if in_run:
                runs.append((s, idx)); in_run = False
    if in_run:
        runs.append((s, len(input_ids)))
    return runs

## LLaDA-Instruct infill

Load the instruction-tuned 8B diffusion LM, mask all chosen cells in one big text, and run masked-diffusion sampling so every blank is filled in a single pass. Decode each mask run and store predictions keyed by `(row, column)`.

In [80]:
# --- LLaDA inference (whole-table infill in a single forward) ---
ll_model = dllm.utils.get_model(model_name_or_path="GSAI-ML/LLaDA-8B-Instruct").eval()
ll_tokenizer = dllm.utils.get_tokenizer(model_name_or_path="GSAI-ML/LLaDA-8B-Instruct")
sampler = MDLMSampler(model=ll_model, tokenizer=ll_tokenizer)

text, keys = render_for_llada(candidates_readable, manual_masks, ll_tokenizer, max_desc_words=40)
input_ids = ll_tokenizer.encode(text, add_special_tokens=True)
print(f"Input length: {len(input_ids)} tokens")

mask_id = ll_tokenizer.mask_token_id
runs = find_mask_runs(input_ids, mask_id)
assert len(runs) == len(keys), f"runs {len(runs)} != keys {len(keys)} — tokenization drift; lower max_desc_words"
n_mask_tokens = sum(end - start for start, end in runs)
print(f"Masked cells: {len(keys)}, mask tokens: {n_mask_tokens}")

cfg = MDLMSamplerConfig(steps=n_mask_tokens, block_size=None, return_dict=True)
out = sampler.infill([input_ids], cfg)
seq = out.sequences[0].tolist()

llada_preds = {}
for (start, end), (row_i, col, orig) in zip(runs, keys):
    pred = ll_tokenizer.decode(seq[start:end]).strip()
    llada_preds[(row_i, col)] = pred

print("\nSample LLaDA predictions:")
for (row_i, col), pred in list(llada_preds.items())[:8]:
    orig = next(o for (i, c, o) in keys if i == row_i and c == col)
    print(f"  Row {row_i} {col}: pred={pred!r}  orig={orig!r}")

del ll_model, sampler
torch.cuda.empty_cache()
gc.collect()

Input length: 3707 tokens
Masked cells: 32, mask tokens: 32

Sample LLaDA predictions:
  Row 0 Gender: pred='Male'  orig='Male'
  Row 0 MaturitySize: pred='Medium'  orig='Small'
  Row 1 Color1: pred='Black'  orig='Brown'
  Row 1 FurLength: pred='Short'  orig='Medium'
  Row 2 Gender: pred='Male'  orig='Male'
  Row 3 MaturitySize: pred='Medium'  orig='Medium'
  Row 3 Vaccinated: pred='Yes'  orig='No'
  Row 4 FurLength: pred='Medium'  orig='Medium'


485

## Step-through viewer (Instruct)

Build a tabular viewer over the saved denoising history. Cells turn green the step they get committed; still-masked cells show `_`; unmasked cells show their original known value.

In [81]:
# --- Sense-check: step through LLaDA-Instruct denoising (tabular view) ---
# green = committed this step, grey = still masked, plain = original (known) value
# Call tv_instruct.next() in a new cell to advance one step at a time
import importlib, dllm.utils.visualizers, dllm.utils
importlib.reload(dllm.utils.visualizers)
importlib.reload(dllm.utils)
from dllm.utils import TableDiffViewer

tv_instruct = TableDiffViewer(out.histories, ll_tokenizer, runs, keys, rows_df=candidates_readable)

Advance one diffusion step.

In [82]:
tv_instruct.next()

Step 0/32  |  filled 0/32  |  committed this step: 0


,Gender,MaturitySize,Color1,FurLength,Vaccinated,Age
row,,,,,,
0,_,_,Black,Short,No,3
1,Male,Medium,_,_,Yes,1
2,_,Medium,Black,Short,No,1
3,Male,_,Black,Long,_,12
4,Female,Medium,Black,_,Not Sure,12
5,_,Large,Black,Long,Yes,_
6,Male,Medium,_,Long,Yes,2
7,Mixed,_,Black,Medium,No,2
8,_,Medium,Brown,_,Not Sure,3


## LLaDA-Base infill

Same task with the non-instruction-tuned base model — useful to see whether instruction tuning matters for tabular infill.

In [83]:
# --- LLaDA-8B-Base (same infill task, no instruction tuning) ---
base_model = dllm.utils.get_model(model_name_or_path="GSAI-ML/LLaDA-8B-Base").eval()
base_tokenizer = dllm.utils.get_tokenizer(model_name_or_path="GSAI-ML/LLaDA-8B-Base")
base_sampler = MDLMSampler(model=base_model, tokenizer=base_tokenizer)

base_text, base_keys = render_for_llada(candidates_readable, manual_masks, base_tokenizer, max_desc_words=40)
base_input_ids = base_tokenizer.encode(base_text, add_special_tokens=True)
print(f"Base input length: {len(base_input_ids)} tokens")

base_mask_id = base_tokenizer.mask_token_id
base_runs = find_mask_runs(base_input_ids, base_mask_id)
assert len(base_runs) == len(base_keys), f"runs {len(base_runs)} != keys {len(base_keys)}"
base_n_masks = sum(end - start for start, end in base_runs)
print(f"Masked cells: {len(base_keys)}, mask tokens: {base_n_masks}")

base_cfg = MDLMSamplerConfig(steps=base_n_masks, block_size=None, return_dict=True)
base_out = base_sampler.infill([base_input_ids], base_cfg)
base_seq = base_out.sequences[0].tolist()

llada_base_preds = {}
for (start, end), (row_i, col, orig) in zip(base_runs, base_keys):
    pred = base_tokenizer.decode(base_seq[start:end]).strip()
    llada_base_preds[(row_i, col)] = pred

print("\nSample LLaDA-Base predictions:")
for (row_i, col), pred in list(llada_base_preds.items())[:8]:
    orig = next(o for (i, c, o) in base_keys if i == row_i and c == col)
    print(f"  Row {row_i} {col}: pred={pred!r}  orig={orig!r}")

del base_model, base_sampler
torch.cuda.empty_cache()
gc.collect()

Base input length: 3707 tokens
Masked cells: 32, mask tokens: 32

Sample LLaDA-Base predictions:
  Row 0 Gender: pred='Male'  orig='Male'
  Row 0 MaturitySize: pred='Small'  orig='Small'
  Row 1 Color1: pred='Black'  orig='Brown'
  Row 1 FurLength: pred='Short'  orig='Medium'
  Row 2 Gender: pred='Male'  orig='Male'
  Row 3 MaturitySize: pred='Small'  orig='Medium'
  Row 3 Vaccinated: pred='Yes'  orig='No'
  Row 4 FurLength: pred='Short'  orig='Medium'


119

## Step-through viewer (Base)

In [84]:
# --- Sense-check: step through LLaDA-Base denoising (tabular view) ---
tv_base = TableDiffViewer(base_out.histories, base_tokenizer, base_runs, base_keys, rows_df=candidates_readable)
tv_base.next()

Step 0/32  |  filled 0/32  |  committed this step: 0


,Gender,MaturitySize,Color1,FurLength,Vaccinated,Age
row,,,,,,
0,_,_,Black,Short,No,3
1,Male,Medium,_,_,Yes,1
2,_,Medium,Black,Short,No,1
3,Male,_,Black,Long,_,12
4,Female,Medium,Black,_,Not Sure,12
5,_,Large,Black,Long,Yes,_
6,Male,Medium,_,Long,Yes,2
7,Mixed,_,Black,Medium,No,2
8,_,Medium,Brown,_,Not Sure,3


## Llama-3.1 baseline (autoregressive)

A standard left-to-right LM can't natively infill, so we replace masks with `[BLANK_k]` placeholders and ask Llama to output `[BLANK_k]: value` lines. Predictions are parsed back with a regex.

In [85]:
# --- Llama-3.1-8B-Instruct baseline (autoregressive blank-fill) ---
from transformers import AutoModelForCausalLM, AutoTokenizer

LLAMA = "meta-llama/Llama-3.1-8B-Instruct"
llama_tok = AutoTokenizer.from_pretrained(LLAMA)
llama = AutoModelForCausalLM.from_pretrained(LLAMA, torch_dtype=torch.bfloat16, device_map="cuda").eval()

def render_for_llama(rows_df, masks_by_row, max_desc_words=40):
    parts, blank_keys = [], []
    k = 1
    for i, row in rows_df.iterrows():
        parts.append(f"\n=== Row {i+1} ===\n")
        masked_cols = set(masks_by_row.get(i, []))
        for f in RENDER_FIELDS:
            val = str(row[f])
            if f == "Age":
                try: val = str(int(float(val)))
                except: pass
            if f == "Description":
                words = val.split()
                if len(words) > max_desc_words:
                    val = " ".join(words[:max_desc_words]) + "..."
            if f in masked_cols:
                parts.append(f"{f}: [BLANK_{k}]\n")
                blank_keys.append((i, f, val))
                k += 1
            else:
                parts.append(f"{f}: {val}\n")
    return "".join(parts), blank_keys

table_text, blank_keys = render_for_llama(candidates_readable, manual_masks)
prompt = (
    "You are filling in missing fields in rows from a pet adoption dataset. "
    "Each row's Description is intact and provides context. Blanks are labelled "
    "[BLANK_1], [BLANK_2], etc.\n\n"
    f"Table:\n{table_text}\n\n"
    f"Output exactly {len(blank_keys)} lines, one per blank, in this format:\n"
    "[BLANK_1]: <value>\n[BLANK_2]: <value>\n...\n"
    "Use only short answers like 'Male', 'Medium', 'Yes', 'Black', '24'. "
    "Do not include any explanation or other text."
)
messages = [{"role": "user", "content": prompt}]
inp = llama_tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
with torch.no_grad():
    gen = llama.generate(inp, max_new_tokens=12 * len(blank_keys), do_sample=False)
response = llama_tok.decode(gen[0, inp.shape[1]:], skip_special_tokens=True)

llama_preds = {}
pat = re.compile(r"\[BLANK_(\d+)\]\s*:\s*(.+)")
for line in response.splitlines():
    m = pat.search(line)
    if m:
        k = int(m.group(1))
        if 1 <= k <= len(blank_keys):
            row_i, col, _ = blank_keys[k - 1]
            llama_preds[(row_i, col)] = m.group(2).strip().rstrip(".,")
for (i, c, _) in blank_keys:
    llama_preds.setdefault((i, c), "")

print(f"Llama parsed {sum(1 for v in llama_preds.values() if v)}/{len(blank_keys)} blanks")

del llama, llama_tok
torch.cuda.empty_cache()
gc.collect()

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Llama parsed 32/32 blanks


11

## TabPFN baseline (purely tabular)

A non-LM control: train a TabPFN classifier (or regressor for Age) per masked column using the other columns as features. Sees zero of the `Description` text — purely a tabular prior over the dataset.

In [86]:

from tabpfn import TabPFNClassifier, TabPFNRegressor

TABPFN_FEATURES = ["Type","Age","Breed1","Breed2","Gender","Color1","Color2","Color3",
                   "MaturitySize","FurLength","Vaccinated","Dewormed","Sterilized",
                   "Health","Quantity","Fee","State"]

DECODE_FOR_COL = {
    "Gender": GENDER_MAP, "MaturitySize": SIZE_MAP, "FurLength": FUR_MAP,
    "Vaccinated": YESNO_MAP, "Sterilized": YESNO_MAP, "Health": HEALTH_MAP,
    "Color1": color_map,
}

# Group masked cells by column
by_col = {}
for row_i, cols in manual_masks.items():
    for c in cols:
        by_col.setdefault(c, []).append(row_i)

fit_pool = train.sample(n=min(8000, len(train)), random_state=0).reset_index(drop=True)

tabpfn_preds = {}
for col, row_indices in by_col.items():
    feats = [c for c in TABPFN_FEATURES if c != col]
    X_train = fit_pool[feats].values
    y_train = fit_pool[col].values
    X_test  = candidates.iloc[row_indices][feats].values

    if col == "Age":
        m = TabPFNRegressor()
    else:
        m = TabPFNClassifier()
    m.fit(X_train, y_train)
    preds = m.predict(X_test)

    for i, row_i in enumerate(row_indices):
        p = preds[i]
        if col == "Age":
            tabpfn_preds[(row_i, col)] = str(int(round(float(p))))
        else:
            decoder = DECODE_FOR_COL.get(col, {})
            tabpfn_preds[(row_i, col)] = decoder.get(int(p), str(p))

print(f"TabPFN predictions: {len(tabpfn_preds)}")
for k, v in list(tabpfn_preds.items())[:8]:
    print(f"  {k}: {v!r}")

TabPFN predictions: 32
  (0, 'Gender'): 'Male'
  (2, 'Gender'): 'Female'
  (5, 'Gender'): 'Female'
  (8, 'Gender'): 'Female'
  (12, 'Gender'): 'Mixed'
  (14, 'Gender'): 'Female'
  (18, 'Gender'): 'Female'
  (20, 'Gender'): 'Mixed'


## Evaluation

Per-column accuracy for categorical fields, MAE for Age. Plus a few worked rows so you can eyeball where each method agrees or disagrees with the ground truth.

In [87]:
# --- Evaluation: accuracy for categorical, MAE for Age ---
def norm(s): return str(s).strip().lower().rstrip(".,")

# Ground truth (readable strings, original Age)
gt = {}
for row_i, cols in manual_masks.items():
    for c in cols:
        if c == "Age":
            gt[(row_i, c)] = str(int(float(candidates.loc[row_i, "Age"])))
        else:
            gt[(row_i, c)] = str(candidates_readable.loc[row_i, c])

models = {
    "LLaDA-Instruct": llada_preds,
    "LLaDA-Base":     llada_base_preds,
    "Llama":          llama_preds,
    "TabPFN":         tabpfn_preds,
}
all_cols = sorted({c for cols in manual_masks.values() for c in cols})

rows_out = {}
for name, preds in models.items():
    rows_out[name] = {}
    for c in all_cols:
        ks = [k for k in gt if k[1] == c]
        if c == "Age":
            errs = []
            for k in ks:
                try: errs.append(abs(float(preds.get(k, "")) - float(gt[k])))
                except: errs.append(np.nan)
            rows_out[name][c] = f"MAE={np.nanmean(errs):.1f}" if errs else "—"
        else:
            hits = sum(norm(preds.get(k, "")) == norm(gt[k]) for k in ks)
            rows_out[name][c] = f"{hits}/{len(ks)} ({hits/len(ks):.0%})" if ks else "—"

results_df = pd.DataFrame(rows_out).T[all_cols]
print("=== Results (per-field per-model) ===")
print(results_df.to_string())

print("\n=== Worked examples ===")
for k in list(gt.keys())[:6]:
    row_i, col = k
    print(f"\nRow {row_i+1}  {col}:  truth={gt[k]!r}")
    for name in models:
        print(f"    {name:15s} {models[name].get(k, '')!r}")

=== Results (per-field per-model) ===
                     Age     Color1  FurLength     Gender MaturitySize Vaccinated
LLaDA-Instruct   MAE=1.0  1/6 (17%)  2/6 (33%)  6/9 (67%)    4/7 (57%)  1/3 (33%)
LLaDA-Base       MAE=2.0  1/6 (17%)  1/6 (17%)  5/9 (56%)    3/7 (43%)  1/3 (33%)
Llama            MAE=2.0  2/6 (33%)  1/6 (17%)  6/9 (67%)    1/7 (14%)  1/3 (33%)
TabPFN          MAE=10.0  4/6 (67%)  2/6 (33%)  5/9 (56%)    4/7 (57%)  2/3 (67%)

=== Worked examples ===

Row 1  Gender:  truth='Male'
    LLaDA-Instruct  'Male'
    LLaDA-Base      'Male'
    Llama           'Male'
    TabPFN          'Male'

Row 1  MaturitySize:  truth='Small'
    LLaDA-Instruct  'Medium'
    LLaDA-Base      'Small'
    Llama           'Medium'
    TabPFN          'Medium'

Row 2  Color1:  truth='Brown'
    LLaDA-Instruct  'Black'
    LLaDA-Base      'Black'
    Llama           'Brown'
    TabPFN          'Brown'

Row 2  FurLength:  truth='Medium'
    LLaDA-Instruct  'Short'
    LLaDA-Base      'Short'
   